# Discovery + Silver: `crm.accounts`

Primera tabla del dominio `crm`. Sin FKs (es la raiz del dominio comercial: empresas/organizaciones).

In [1]:
import sys
from pathlib import Path
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine, get_psycopg2_connection

engine = get_engine()
SQL_SILVER = Path("/home/jovyan/work/sql/silver")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

df = pd.read_sql("SELECT * FROM bronze.crm__accounts", engine)
df.shape

(5000, 10)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

account_id                object
name                      object
industry                  object
country                   object
annual_revenue            object
employees                 object
created_at                object
_source_file              object
_ingested_at      datetime64[ns]
_dag_run_id               object
dtype: object


,account_id,name,industry,country,annual_revenue,employees,created_at,_source_file,_ingested_at,_dag_run_id
0,ACC-0000001,Patagonia Group,services,ES,342518.86,3,2018-12-02 14:00:10,crm/accounts.csv,2026-07-21 09:50:01.845346,manual__2026-07-21T09:49:58+00:00
1,ACC-0000002,Rio SpA,energy,BR,710384.81,73,2020-09-21 21:59:21,crm/accounts.csv,2026-07-21 09:50:01.845346,manual__2026-07-21T09:49:58+00:00
2,ACC-0000003,Litio Foods,tech,BR,1295021.77,354,2021-06-11 22:57:54,crm/accounts.csv,2026-07-21 09:50:01.845346,manual__2026-07-21T09:49:58+00:00
3,ACC-0000004,Norte Mining,energy,AR,195949.54,10,2025-09-21 12:05:37,crm/accounts.csv,2026-07-21 09:50:01.845346,manual__2026-07-21T09:49:58+00:00
4,ACC-0000005,Atacama Industries,services,CO,235139.09,81,2020-01-14 22:51:10,crm/accounts.csv,2026-07-21 09:50:01.845346,manual__2026-07-21T09:49:58+00:00


## 2. Nulos y duplicados

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("account_id duplicados:", df["account_id"].duplicated().sum())

Nulos por columna:
account_id        0
name              0
industry          0
country           0
annual_revenue    0
employees         0
created_at        0
_source_file      0
_ingested_at      0
_dag_run_id       0
dtype: int64

account_id duplicados: 0


## 3. `industry`, `country`, `annual_revenue`, `employees`

In [4]:
print("industry:")
print(df["industry"].value_counts())
print()

revenue = pd.to_numeric(df["annual_revenue"], errors="coerce")
employees = pd.to_numeric(df["employees"], errors="coerce")
print("annual_revenue <= 0:", (revenue <= 0).sum())
print("employees <= 0:", (employees <= 0).sum())

industry:
industry
tech             909
finance          751
retail           739
health           604
manufacturing    579
education        527
services         495
energy           396
Name: count, dtype: int64

annual_revenue <= 0: 0
employees <= 0: 0


## 4. Conclusion

Tabla limpia (sin nulos, sin duplicados, valores positivos). Solo tipado y estandarizacion.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["account_id", "name", "industry", "country", "annual_revenue", "employees", "created_at"]].copy()

df_silver["name"] = df_silver["name"].str.strip()
df_silver["industry"] = df_silver["industry"].str.strip().str.lower()
df_silver["country"] = df_silver["country"].str.strip().str.upper()
df_silver["annual_revenue"] = pd.to_numeric(df_silver["annual_revenue"], errors="raise")
df_silver["employees"] = pd.to_numeric(df_silver["employees"], errors="raise").astype(int)
df_silver["created_at"] = pd.to_datetime(df_silver["created_at"])

df_silver.head()

,account_id,name,industry,country,annual_revenue,employees,created_at
0,ACC-0000001,Patagonia Group,services,ES,342518.86,3,2018-12-02 14:00:10
1,ACC-0000002,Rio SpA,energy,BR,710384.81,73,2020-09-21 21:59:21
2,ACC-0000003,Litio Foods,tech,BR,1295021.77,354,2021-06-11 22:57:54
3,ACC-0000004,Norte Mining,energy,AR,195949.54,10,2025-09-21 12:05:37
4,ACC-0000005,Atacama Industries,services,CO,235139.09,81,2020-01-14 22:51:10


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["account_id"].is_unique
assert df_silver["annual_revenue"].gt(0).all()
assert df_silver["employees"].gt(0).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 5000 filas listas para silver


## 7. Escribir en `silver.crm__accounts`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

run_sql_file(SQL_SILVER / "crm.sql")

conn = get_psycopg2_connection()
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE silver.crm__accounts CASCADE;")
conn.commit()
conn.close()

df_silver.to_sql(
    "crm__accounts",
    engine,
    schema="silver",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=2000,
)
print("Escrito en silver.crm__accounts")

OK: crm.sql ejecutado


Escrito en silver.crm__accounts


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.crm__accounts LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT account_id) AS ids_unicos FROM silver.crm__accounts", engine))
check

   filas  ids_unicos
0   5000        5000


,account_id,name,industry,country,annual_revenue,employees,created_at,_silver_loaded_at
0,ACC-0000001,Patagonia Group,services,ES,342518.86,3,2018-12-02 14:00:10,2026-07-21 09:50:19.160025+00:00
1,ACC-0000002,Rio SpA,energy,BR,710384.81,73,2020-09-21 21:59:21,2026-07-21 09:50:19.160025+00:00
2,ACC-0000003,Litio Foods,tech,BR,1295021.77,354,2021-06-11 22:57:54,2026-07-21 09:50:19.160025+00:00
3,ACC-0000004,Norte Mining,energy,AR,195949.54,10,2025-09-21 12:05:37,2026-07-21 09:50:19.160025+00:00
4,ACC-0000005,Atacama Industries,services,CO,235139.09,81,2020-01-14 22:51:10,2026-07-21 09:50:19.160025+00:00
